[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isrunej/Modul_Kinematik_Robot/blob/main/Modul_02_IK_2DOF.ipynb)

# Modul 2: Inverse Kinematics — Robot 2-DOF Planar
## Studi Kasus: Robot Pemetik Stroberi — Menjangkau Target

---

### Tujuan Pembelajaran
1. Memahami perbedaan FK dan IK secara konseptual
2. Menyelesaikan IK 2-DOF secara geometrik dan analitik
3. Memahami konsep **dua solusi** pada IK dan maknanya di lapangan
4. Mengimplementasikan IK dalam Python untuk robot pemetik stroberi

---

### Konteks: Masalah Nyata di Greenhouse

Di Modul 1, kita memberi tahu robot: *"set sudut joint ke 45° dan -30°"*, lalu kita hitung posisi tangannya.

**Masalah:** Di dunia nyata, kita **tidak tahu sudutnya**. Yang kita tahu adalah **posisi stroberi** dari sensor kamera!

Kamera mendeteksi stroberi di koordinat $(x, y)$. Kita butuh tahu: **sudut berapa yang harus diperintahkan ke motor?**

```
  POSISI STROBERI (x, y)  →  [INVERSE KINEMATICS]  →  SUDUT JOINT (θ₁, θ₂)
```

Inilah **Inverse Kinematics (IK)**.

---
## Bagian 1: Penurunan Rumus IK secara Geometrik

### Langkah 1: Cari θ₂ (Elbow Angle)

Kita tahu posisi target $(x, y)$. Jarak dari base ke target:
$$r = \sqrt{x^2 + y^2}$$

Gunakan **Hukum Cosinus** pada segitiga yang dibentuk oleh Link 1, Link 2, dan jarak $r$:

$$r^2 = L_1^2 + L_2^2 - 2 L_1 L_2 \cos(\pi - \theta_2)$$

$$\cos(\theta_2) = \frac{x^2 + y^2 - L_1^2 - L_2^2}{2 L_1 L_2}$$

### Langkah 2: Cari θ₁

$$\theta_1 = \text{atan2}(y, x) - \text{atan2}(L_2 \sin\theta_2,\ L_1 + L_2\cos\theta_2)$$

### Dua Solusi (Elbow-Up vs Elbow-Down)

Karena $\sin^2 + \cos^2 = 1$, ada **dua nilai** $\theta_2$:
- $\theta_2 = +\arccos(c_2)$ → konfigurasi **Elbow-Down** (siku menekuk ke bawah)
- $\theta_2 = -\arccos(c_2)$ → konfigurasi **Elbow-Up** (siku menekuk ke atas)

Di greenhouse, mana yang lebih aman? **Tergantung rintangan!**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Parameter robot (sama dengan Modul 1)
L1 = 0.4  # meter
L2 = 0.3  # meter

print("Parameter robot siap!")
print(f"L1 = {L1} m, L2 = {L2} m")
print(f"Jangkauan maksimal = {L1+L2} m")
print(f"Jangkauan minimal  = {abs(L1-L2)} m")

In [ ]:
# ============================================
# FUNGSI INVERSE KINEMATICS
# ============================================

def inverse_kinematics_2dof(x_target, y_target, L1, L2):
    """
    Hitung sudut joint untuk mencapai posisi target.
    
    Input:
        x_target, y_target : koordinat target (meter)
        L1, L2             : panjang link (meter)
    
    Output:
        dict dengan dua solusi: 'elbow_down' dan 'elbow_up'
        Masing-masing berisi (theta1, theta2) dalam derajat
        Atau None jika target tidak terjangkau
    """
    
    # Jarak ke target
    r = np.sqrt(x_target**2 + y_target**2)
    
    # Cek apakah target bisa dijangkau
    if r > L1 + L2:
        print(f"❌ Target ({x_target}, {y_target}) TERLALU JAUH! r={r:.3f} > {L1+L2}")
        return None
    if r < abs(L1 - L2):
        print(f"❌ Target ({x_target}, {y_target}) TERLALU DEKAT! r={r:.3f} < {abs(L1-L2)}")
        return None
    
    # Hitung cos(θ₂) dari Hukum Cosinus
    cos_theta2 = (x_target**2 + y_target**2 - L1**2 - L2**2) / (2 * L1 * L2)
    
    # Klem nilai agar tidak keluar rentang [-1, 1] akibat floating point
    cos_theta2 = np.clip(cos_theta2, -1, 1)
    
    # Dua kemungkinan θ₂
    theta2_elbow_down =  np.arccos(cos_theta2)   # positif = elbow down
    theta2_elbow_up   = -np.arccos(cos_theta2)   # negatif = elbow up
    
    def hitung_theta1(theta2):
        # θ₁ = atan2(y,x) - atan2(L2 sin θ₂, L1 + L2 cos θ₂)
        alpha = np.arctan2(y_target, x_target)
        beta  = np.arctan2(L2 * np.sin(theta2), L1 + L2 * np.cos(theta2))
        return alpha - beta
    
    theta1_elbow_down = hitung_theta1(theta2_elbow_down)
    theta1_elbow_up   = hitung_theta1(theta2_elbow_up)
    
    return {
        'elbow_down': (
            np.degrees(theta1_elbow_down),
            np.degrees(theta2_elbow_down)
        ),
        'elbow_up': (
            np.degrees(theta1_elbow_up),
            np.degrees(theta2_elbow_up)
        )
    }

# --- Tes dengan posisi stroberi ---
x_stroberi = 0.5
y_stroberi = 0.3

solusi = inverse_kinematics_2dof(x_stroberi, y_stroberi, L1, L2)

if solusi:
    t1_down, t2_down = solusi['elbow_down']
    t1_up,   t2_up   = solusi['elbow_up']
    
    print(f"\n=== INVERSE KINEMATICS ===")
    print(f"Target: stroberi di ({x_stroberi}, {y_stroberi})")
    print(f"\nSolusi 1 (Elbow-Down):")
    print(f"  θ₁ = {t1_down:.2f}°,  θ₂ = {t2_down:.2f}°")
    print(f"\nSolusi 2 (Elbow-Up):")
    print(f"  θ₁ = {t1_up:.2f}°,  θ₂ = {t2_up:.2f}°")

---
## Bagian 2: Verifikasi — Apakah Sudut Hasil IK Benar?

In [ ]:
def forward_kinematics_2dof(theta1, theta2, L1, L2):
    theta1_rad = np.radians(theta1)
    theta2_rad = np.radians(theta2)
    x0, y0 = 0, 0
    x1 = L1 * np.cos(theta1_rad)
    y1 = L1 * np.sin(theta1_rad)
    x2 = x1 + L2 * np.cos(theta1_rad + theta2_rad)
    y2 = y1 + L2 * np.sin(theta1_rad + theta2_rad)
    return {'base': (x0,y0), 'joint2': (x1,y1), 'end_effector': (x2,y2)}

# Verifikasi Solusi 1
hasil_fk_down = forward_kinematics_2dof(t1_down, t2_down, L1, L2)
x_fk, y_fk = hasil_fk_down['end_effector']
print(f"Verifikasi Solusi 1 (Elbow-Down):")
print(f"  Target  : ({x_stroberi}, {y_stroberi})")
print(f"  FK hasil: ({x_fk:.4f}, {y_fk:.4f})")
error = np.sqrt((x_fk-x_stroberi)**2 + (y_fk-y_stroberi)**2)
print(f"  Error   : {error:.8f} m  {'✅ OK' if error < 1e-6 else '❌ Error besar'}")

# Verifikasi Solusi 2
hasil_fk_up = forward_kinematics_2dof(t1_up, t2_up, L1, L2)
x_fk2, y_fk2 = hasil_fk_up['end_effector']
print(f"\nVerifikasi Solusi 2 (Elbow-Up):")
print(f"  Target  : ({x_stroberi}, {y_stroberi})")
print(f"  FK hasil: ({x_fk2:.4f}, {y_fk2:.4f})")
error2 = np.sqrt((x_fk2-x_stroberi)**2 + (y_fk2-y_stroberi)**2)
print(f"  Error   : {error2:.8f} m  {'✅ OK' if error2 < 1e-6 else '❌ Error besar'}")

---
## Bagian 3: Visualisasi Dua Solusi IK

In [ ]:
def gambar_robot(ax, posisi, warna_link1='blue', warna_link2='green', 
                 label_suffix='', alpha=1.0):
    x0, y0 = posisi['base']
    x1, y1 = posisi['joint2']
    x2, y2 = posisi['end_effector']
    ax.plot([x0,x1],[y0,y1], color=warna_link1, linewidth=6, 
            solid_capstyle='round', alpha=alpha, label=f'Link 1{label_suffix}')
    ax.plot([x1,x2],[y1,y2], color=warna_link2, linewidth=5, 
            solid_capstyle='round', alpha=alpha, label=f'Link 2{label_suffix}')
    ax.plot(x0, y0, 'ko', markersize=14, alpha=alpha, zorder=5)
    ax.plot(x0, y0, 'wo', markersize=6, zorder=6)
    ax.plot(x1, y1, 'ko', markersize=12, alpha=alpha, zorder=5)
    ax.plot(x1, y1, 'wo', markersize=5, zorder=6)
    ax.plot(x2, y2, 'r*', markersize=16, alpha=alpha, zorder=5)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (konfigurasi, (t1, t2)) in enumerate([
    ('Elbow-Down', (t1_down, t2_down)),
    ('Elbow-Up',   (t1_up,   t2_up))
]):
    ax = axes[idx]
    
    posisi = forward_kinematics_2dof(t1, t2, L1, L2)
    gambar_robot(ax, posisi)
    
    # Gambar target stroberi
    ax.plot(x_stroberi, y_stroberi, marker='*', color='red', 
            markersize=25, zorder=10)
    ax.annotate(f'Stroberi\n({x_stroberi}, {y_stroberi})',
                (x_stroberi, y_stroberi), 
                textcoords='offset points', xytext=(10, 5), 
                fontsize=10, color='darkred', fontweight='bold')
    
    # Workspace
    ws = plt.Circle((0,0), L1+L2, fill=False, 
                    color='gray', linestyle='--', alpha=0.3)
    ax.add_patch(ws)
    
    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-0.15, 0.85)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.set_title(f'Solusi {idx+1}: {konfigurasi}\nθ₁={t1:.1f}°, θ₂={t2:.1f}°', 
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('x (meter)')
    ax.set_ylabel('y (meter)')
    
    # Info kotak
    info = f'θ₁ = {t1:.2f}°\nθ₂ = {t2:.2f}°'
    ax.text(0.03, 0.97, info, transform=ax.transAxes,
            va='top', fontsize=11,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Dua Solusi IK untuk Target yang Sama\n'
             '(Keduanya benar secara matematis — pilihan tergantung kondisi greenhouse)', 
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Bagian 4: Simulasi Pemetikan — Lintasan Robot

In [ ]:
def simulasi_pemetikan(daftar_stroberi, L1, L2, konfigurasi='elbow_down'):
    """
    Simulasikan robot bergerak memetik beberapa stroberi secara berurutan.
    Menampilkan satu frame per stroberi.
    """
    n = len(daftar_stroberi)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*4.5))
    if n == 1:
        axes = [[axes]]
    elif rows == 1:
        axes = [axes]
    
    for i, (x_t, y_t) in enumerate(daftar_stroberi):
        row = i // cols
        col = i %  cols
        ax  = axes[row][col]
        
        solusi = inverse_kinematics_2dof(x_t, y_t, L1, L2)
        
        if solusi is None:
            ax.set_title(f'Stroberi {i+1}: TIDAK TERJANGKAU', color='red')
            ax.axis('off')
            continue
        
        t1, t2 = solusi[konfigurasi]
        posisi = forward_kinematics_2dof(t1, t2, L1, L2)
        gambar_robot(ax, posisi)
        
        # Tandai semua stroberi
        for j, (xs, ys) in enumerate(daftar_stroberi):
            warna = 'red' if j == i else 'orange'
            ukuran = 25 if j == i else 12
            ax.plot(xs, ys, '*', color=warna, markersize=ukuran, zorder=10)
            if j == i:
                ax.annotate(f'TARGET', (xs, ys),
                            textcoords='offset points', xytext=(5, 8),
                            fontsize=8, color='darkred', fontweight='bold')
        
        ws = plt.Circle((0,0), L1+L2, fill=False, 
                        color='gray', linestyle='--', alpha=0.2)
        ax.add_patch(ws)
        ax.set_xlim(-0.8, 0.8)
        ax.set_ylim(-0.1, 0.8)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_title(f'Langkah {i+1}: Petik Stroberi di ({x_t}, {y_t})\n'
                     f'θ₁={t1:.1f}°, θ₂={t2:.1f}°', fontsize=10)
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
    
    # Sembunyikan panel kosong
    for i in range(n, rows*cols):
        axes[i//cols][i%cols].set_visible(False)
    
    plt.suptitle(f'Simulasi Robot Pemetik Stroberi ({konfigurasi.replace("_"," ").title()})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Daftar posisi stroberi yang terdeteksi kamera
posisi_stroberi = [
    (0.55, 0.25),   # stroberi 1
    (0.40, 0.50),   # stroberi 2  
    (0.15, 0.60),   # stroberi 3
    (-0.10, 0.55),  # stroberi 4
    (0.65, 0.10),   # stroberi 5
    (0.90, 0.10),   # stroberi 6 — coba: apakah terjangkau?
]

simulasi_pemetikan(posisi_stroberi, L1, L2, konfigurasi='elbow_down')

---
## Bagian 5: Memilih Solusi Terbaik

Jika ada dua solusi IK, bagaimana robot memilih? Beberapa strategi:

1. **Hindari singularitas** (posisi ekstrem) → pilih sudut yang lebih 'tengah'
2. **Minimum gerakan** → pilih solusi yang perubahan sudutnya paling kecil dari posisi saat ini
3. **Hindari rintangan** → pilih konfigurasi yang tidak menabrak tanaman

Di bawah ini kita implementasikan strategi **minimum gerakan**:

In [ ]:
def pilih_solusi_terbaik(solusi, theta1_saat_ini, theta2_saat_ini):
    """
    Pilih solusi IK yang membutuhkan pergerakan sudut paling sedikit.
    (Hemat energi motor!)
    """
    if solusi is None:
        return None
    
    gerakan_down = (abs(solusi['elbow_down'][0] - theta1_saat_ini) + 
                    abs(solusi['elbow_down'][1] - theta2_saat_ini))
    gerakan_up   = (abs(solusi['elbow_up'][0]   - theta1_saat_ini) + 
                    abs(solusi['elbow_up'][1]   - theta2_saat_ini))
    
    if gerakan_down <= gerakan_up:
        return 'elbow_down', solusi['elbow_down']
    else:
        return 'elbow_up', solusi['elbow_up']

# Simulasi sekuensial: robot mulai dari posisi awal (home)
t1_sekarang = 90   # posisi awal: lengan tegak lurus
t2_sekarang =  0

print("=== SIMULASI PEMILIHAN JALUR OPTIMAL ===")
print(f"Posisi awal: θ₁={t1_sekarang}°, θ₂={t2_sekarang}°\n")

for i, (xt, yt) in enumerate(posisi_stroberi[:5]):
    solusi = inverse_kinematics_2dof(xt, yt, L1, L2)
    if solusi:
        konfigurasi, (t1_baru, t2_baru) = pilih_solusi_terbaik(
            solusi, t1_sekarang, t2_sekarang
        )
        delta = abs(t1_baru-t1_sekarang) + abs(t2_baru-t2_sekarang)
        print(f"Stroberi {i+1} ({xt}, {yt}):")
        print(f"  → Pilih: {konfigurasi} | θ₁={t1_baru:.1f}°, θ₂={t2_baru:.1f}° | Δ gerakan={delta:.1f}°")
        t1_sekarang = t1_baru
        t2_sekarang = t2_baru

---
## Ringkasan Modul 2

| Konsep | Rumus |
|--------|-------|
| $\cos\theta_2$ | $\dfrac{x^2+y^2-L_1^2-L_2^2}{2L_1L_2}$ |
| $\theta_2$ | $\pm\arccos(c_2)$ → **dua solusi** |
| $\theta_1$ | $\text{atan2}(y,x) - \text{atan2}(L_2\sin\theta_2,\ L_1+L_2\cos\theta_2)$ |

**Poin kunci:**
- IK = "diberi posisi target → cari sudut joint"
- Selalu ada **dua solusi** (elbow-up vs elbow-down) untuk robot 2-DOF
- Target di luar workspace = **tidak ada solusi**
- Strategi memilih solusi: minimum gerakan atau hindari rintangan

---
## Lanjut ke Modul 3: Kinematik 3-DOF Spatial

Robot greenhouse nyata bergerak di **ruang 3D** — baris tanaman panjang, ketinggian berbeda. Modul berikutnya: FK dan IK untuk robot 3-DOF!